# Market Data Collection Runbook

This notebook collects stock and market data for `SPY`, `QQQ`, `RSP`, and `^VIX`. It does not collect GDELT or news/headline data.

Raw market prices are written under `data/raw/market/` and are never overwritten. The merged daily market panel is generated under `data/processed/panel_daily.csv`, and replacing it requires `OVERWRITE_PANEL = True`.

This notebook is a runbook, not core pipeline logic. It calls functions from `src/mci/market_data.py`.

## Parameters

In [ ]:
from datetime import date
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
START_DATE = date(2022, 1, 1)
END_DATE = date(2026, 5, 31)
SYMBOLS = ("SPY", "QQQ", "RSP", "^VIX")

RUN_MARKET_DATA = True
BUILD_PANEL = True
OVERWRITE_PANEL = False
MARKET_PRICES_CSV = None
RUN_MARKET_PREFLIGHT = False
MARKET_MAX_RETRIES = 5
MARKET_BACKOFF_SECONDS = 60.0
MARKET_MAX_BACKOFF_SECONDS = 900.0
MARKET_REQUEST_PAUSE_SECONDS = 5.0

MCI_PATH = Path("data/processed/mci_daily.csv")

## Import And Setup

In [ ]:
import os
import sys

project_root_candidates = []
project_root_override = globals().get("PROJECT_ROOT_OVERRIDE")
if project_root_override:
    project_root_candidates.append(Path(project_root_override).expanduser())

env_project_root = os.environ.get("MCI_PROJECT_ROOT")
if env_project_root:
    project_root_candidates.append(Path(env_project_root).expanduser())

cwd = Path.cwd()
project_root_candidates.extend([cwd, *cwd.parents])
project_root_candidates.extend([
    cwd / "market-criticism-index",
    Path.home() / "Desktop" / "market-criticism-index",
    Path.home() / "market-criticism-index",
])

PROJECT_ROOT = None
seen_candidates = set()
for candidate in project_root_candidates:
    candidate = candidate.resolve()
    if candidate in seen_candidates:
        continue
    seen_candidates.add(candidate)
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "mci").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the project root. Start Jupyter from the repository checkout, set PROJECT_ROOT_OVERRIDE in the Parameters cell, or set MCI_PROJECT_ROOT.")

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from mci.config import PROCESSED_DATA_DIR, RAW_DATA_DIR
from mci.market_data import (
    MarketDataSpec,
    MarketPanelSpec,
    build_market_panel,
    collect_market_data,
    market_data_output_path,
    preflight_market_data_provider,
    validate_market_price_csv,
)

if not MCI_PATH.is_absolute():
    MCI_PATH = PROJECT_ROOT / MCI_PATH

RAW_MARKET_DIR = RAW_DATA_DIR / "market"
PANEL_OUTPUT_PATH = PROCESSED_DATA_DIR / "panel_daily.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw market directory: {RAW_MARKET_DIR}")
print(f"Panel output path: {PANEL_OUTPUT_PATH}")
print(f"MCI path: {MCI_PATH}")

## Collect Raw Market Prices

In [ ]:
expected_prices_path = market_data_output_path(RAW_MARKET_DIR, SYMBOLS, START_DATE, END_DATE)
configured_prices_path = Path(MARKET_PRICES_CSV).expanduser() if MARKET_PRICES_CSV else None
if configured_prices_path is not None and not configured_prices_path.is_absolute():
    configured_prices_path = PROJECT_ROOT / configured_prices_path
market_prices_path = None

print(f"Expected raw market CSV: {expected_prices_path}")

if configured_prices_path is not None:
    try:
        market_prices_path = validate_market_price_csv(configured_prices_path, SYMBOLS)
        print(f"Using validated local market CSV: {market_prices_path}")
    except (OSError, ValueError) as exc:
        print("Configured MARKET_PRICES_CSV is not usable.")
        print(exc)
elif not RUN_MARKET_DATA:
    print("RUN_MARKET_DATA is False; skipping market data collection.")
elif expected_prices_path.exists():
    try:
        market_prices_path = validate_market_price_csv(expected_prices_path, SYMBOLS)
        print("Raw market CSV already exists and validated. Reusing it because raw data is never overwritten.")
    except (OSError, ValueError) as exc:
        print("Existing raw market CSV is not usable.")
        print(exc)
else:
    try:
        market_spec = MarketDataSpec(
            start_date=START_DATE,
            end_date=END_DATE,
            symbols=SYMBOLS,
            raw_output_dir=RAW_MARKET_DIR,
            max_retries=MARKET_MAX_RETRIES,
            backoff_seconds=MARKET_BACKOFF_SECONDS,
            max_backoff_seconds=MARKET_MAX_BACKOFF_SECONDS,
            request_pause_seconds=MARKET_REQUEST_PAUSE_SECONDS,
        )
        if RUN_MARKET_PREFLIGHT:
            print("Running fast provider preflight...")
            preflight_market_data_provider(market_spec)
        market_prices_path = collect_market_data(market_spec)
        print(f"Saved raw market CSV: {market_prices_path}")
    except (FileExistsError, RuntimeError, ValueError) as exc:
        print("Market data collection did not complete.")
        print(exc)
        print("Recovery: retry later, or place a normalized local CSV under data/raw/market/ and set MARKET_PRICES_CSV.")
        if not expected_prices_path.exists():
            market_prices_path = None

## Yahoo 429 Recovery And Local CSV Fallback

HTTP `429` means Yahoo is rate-limiting requests. It does not mean the date range is necessarily wrong. The reproducible path is to use a normalized local CSV under `data/raw/market/` and set `MARKET_PRICES_CSV` in the Parameters cell.

Accepted schema: required columns `date`, `symbol`, `close`; optional columns `open`, `high`, `low`, `adj_close`, `volume`. Accepted symbols are `SPY`, `QQQ`, `RSP`, and `VIX` or `^VIX`.

## Preview Raw Prices

In [ ]:
import pandas as pd

if market_prices_path is None or not Path(market_prices_path).exists():
    print("No raw market CSV is available to preview.")
else:
    raw_prices = pd.read_csv(market_prices_path)
    print(f"Rows: {len(raw_prices):,}")
    if "date" in raw_prices.columns and not raw_prices.empty:
        print(f"Date range: {raw_prices['date'].min()} to {raw_prices['date'].max()}")
    if "symbol" in raw_prices.columns:
        print(f"Symbols: {', '.join(sorted(raw_prices['symbol'].astype(str).unique()))}")

    selected_price = raw_prices.get("close", pd.Series(dtype="object")).astype(str).str.strip()
    if {"symbol", "adj_close"}.issubset(raw_prices.columns):
        adjusted = raw_prices["adj_close"].astype(str).str.strip()
        use_adjusted = raw_prices["symbol"].astype(str).str.upper().isin({"SPY", "QQQ", "RSP"}) & adjusted.ne("")
        selected_price = selected_price.copy()
        selected_price.loc[use_adjusted] = adjusted.loc[use_adjusted]
    missing_selected_price = pd.to_numeric(selected_price.replace("", pd.NA), errors="coerce").isna().sum()
    print(f"Missing selected-price values: {int(missing_selected_price):,}")
    display(raw_prices.head())

## Build Daily Market Panel

In [ ]:
panel_output_path = PANEL_OUTPUT_PATH

if not BUILD_PANEL:
    print("BUILD_PANEL is False; skipping daily panel construction.")
elif market_prices_path is None or not Path(market_prices_path).exists():
    print("Cannot build the panel because no raw market CSV is available.")
elif not MCI_PATH.exists():
    print("Skipping panel build because the MCI CSV is missing.")
    print(f"Expected MCI path: {MCI_PATH}")
    print("Next step: build it with scripts/build_mci_daily.py after headline cleaning is complete.")
else:
    try:
        panel_spec = MarketPanelSpec(
            prices_path=Path(market_prices_path),
            mci_path=MCI_PATH,
            output_path=PANEL_OUTPUT_PATH,
            symbols=SYMBOLS,
            overwrite=OVERWRITE_PANEL,
        )
        panel_output_path = build_market_panel(panel_spec)
        print(f"Saved daily panel: {panel_output_path}")
    except (FileExistsError, RuntimeError, ValueError) as exc:
        print("Panel construction did not complete.")
        print(exc)
        print("Recovery: inspect the message above. Set OVERWRITE_PANEL=True only if you intend to replace data/processed/panel_daily.csv.")

## Preview Panel

In [ ]:
if panel_output_path is None or not Path(panel_output_path).exists():
    print("No panel CSV is available to preview.")
else:
    panel = pd.read_csv(panel_output_path)
    print(f"Rows: {len(panel):,}")
    if "date" in panel.columns and not panel.empty:
        print(f"Date range: {panel['date'].min()} to {panel['date'].max()}")

    key_features = [
        "MCI",
        "mci_rolling_60d_zscore",
        "spy_fwd_log_return_1d",
        "spy_fwd_log_return_5d",
        "spy_fwd_log_return_21d",
        "spy_fwd_realized_vol_21d",
        "spy_fwd_max_drawdown_21d",
        "vix_level",
        "vix_fwd_change_21d",
    ]
    present_features = [column for column in key_features if column in panel.columns]
    print(f"Key feature columns present: {', '.join(present_features)}")
    if present_features:
        display(panel[present_features].isna().sum().rename("missing_values").to_frame())
    display(panel.head())

## Next Steps

If `data/processed/mci_daily.csv` is missing, build it after headline collection, cleaning, and alignment:

```bash
python scripts/build_mci_daily.py --market-csv data/interim/gdelt_all_us_market_20220101_20260531.csv --criticism-csv data/interim/gdelt_candidate_criticism_20220101_20260531.csv --overwrite
```

If `data/processed/panel_daily.csv` exists, run the MVP empirical analysis from the command line:

```bash
python scripts/run_mvp_analysis.py --overwrite
```